# 02 - Aprendizaje de la trayectoria con una red neuronal

Modelamos la trayectoria `(x(t), y(t))` con una red neuronal usando TensorFlow/Keras, sin imponer desde el inicio la forma parabolica.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks, optimizers

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
raw = np.genfromtxt(ROOT / 'data' / 'trajectory_extracted.csv', delimiter=',', names=True)
t = raw['t_s'].astype('float32')
x = raw['x_m'].astype('float32')
y = raw['y_m'].astype('float32')
print(t.shape, x.shape, y.shape)


## Modelo

Usamos un perceptron multicapa pequeno. La entrada es `t` y la salida tiene dos componentes: `x` e `y`.


In [ ]:
keras.utils.set_random_seed(0)
model = keras.Sequential([
    layers.Input(shape=(1,)),
    layers.Dense(32, activation='tanh'),
    layers.Dense(32, activation='tanh'),
    layers.Dense(2)
])

class PrintLoss(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if epoch % 600 == 0:
            print(f'Epoca {epoch:4d} - perdida: {logs["loss"]:.6f}')

model.compile(optimizer=optimizers.Adam(learning_rate=0.01), loss='mse')
history = model.fit(t.reshape(-1,1), np.column_stack([x,y]), epochs=3000, batch_size=len(t), verbose=0, callbacks=[PrintLoss()])


## Evaluacion e interpretacion

Despues del ajuste calculamos la curvatura de `y(t)` con `tf.GradientTape` para estimar `g`.


In [ ]:
preds = model.predict(t.reshape(-1,1), verbose=0)
x_pred, y_pred = preds[:,0], preds[:,1]

t_tf = tf.convert_to_tensor(t.reshape(-1,1), dtype=tf.float32)
with tf.GradientTape() as tape2:
    tape2.watch(t_tf)
    with tf.GradientTape() as tape1:
        tape1.watch(t_tf)
        y_out = model(t_tf, training=False)[:, 1:2]
    dy_dt = tape1.gradient(y_out, t_tf)
d2y_dt2 = tape2.gradient(dy_dt, t_tf)
g_est = -float(tf.reduce_mean(d2y_dt2).numpy())
print(f'Estimacion de g: {g_est:.4f} m/s^2')

fig, axs = plt.subplots(1, 2, figsize=(10,4))
axs[0].plot(t, x, 'o', label='x datos'); axs[0].plot(t, x_pred, '-', label='x red')
axs[1].plot(t, y, 'o', label='y datos'); axs[1].plot(t, y_pred, '-', label='y red')
for ax in axs:
    ax.set_xlabel('t [s]'); ax.grid(True, alpha=0.3); ax.legend()
axs[0].set_ylabel('x [m]'); axs[1].set_ylabel('y [m]')
plt.tight_layout(); plt.show()

plt.figure(figsize=(6,4))
plt.plot(x, y, 'o', label='datos')
plt.plot(x_pred, y_pred, '-', label='red')
plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.gca().set_aspect('equal', adjustable='box')
plt.grid(True, alpha=0.3); plt.legend(); plt.show()


## Preguntas

- Interpolar bien implica recuperar una ley fisica?
- Que pasaria al extrapolar fuera del intervalo observado?
- Como cambiaria el modelo si hubiera resistencia del aire?
